# CARGA A BASE DE DATOS (FASE LOAD)

Cargar el DataFrame maestro generado en la fase de transformación hacia una base de datos SQLite denominada delivery_insights.db. La tabla resultante se almacenará como master_food_data y servirá como fuente para las consultas SQL y el análisis posterior.

In [1]:
## clonar o actualizar repositorio github en colab

from pathlib import Path
import os

ruta_proyecto = Path("/content/etl-python-analisis-food-delivery")

# Si el repositorio no existe, lo clona
if not ruta_proyecto.exists():
    %cd /content
    !git clone https://github.com/dannyturpo-data/etl-python-analisis-food-delivery.git

# Si ya existe, actualiza cambios
else:
    %cd /content/etl-python-analisis-food-delivery
    !git pull origin main

/content
Cloning into 'etl-python-analisis-food-delivery'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 92 (delta 3), reused 3 (delta 1), pack-reused 84 (from 4)
Receiving objects: 100% (92/92), 42.77 MiB | 10.96 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Updating files: 100% (20/20), done.


In [34]:
## configuracion

import sys

if str(ruta_proyecto) not in sys.path:
    sys.path.append(str(ruta_proyecto))

## importando librerias

import pandas as pd
from sqlalchemy import create_engine
from google.colab import files

from IPython.display import display

In [5]:
## funciones propias

from src.exploracion_df import exploratorio_df

## CARGA DE DATAFRAME MAESTRO

In [10]:
df_master = pd.read_csv(
    "/content/etl-python-analisis-food-delivery/data/procesada/df_master.csv"
)

In [11]:
exploratorio_df(df_master)

RESUMEN GENERAL

MUESTRA ALEATORIA


,restaurant_id,position,name_restaurant,score,ratings,category_restaurant,price_range,full_address,lat,lng,calle,ciudad,estado_zip,category_menu,name_menu,description,price
936,1821,4,Mulligans Irish Pub and Grill,4.5,112.0,"european, burgers, pub, family friendly, pizza...",Moderadamente caro,"8933 S 27th St, Franklin, WI, 53132",42.882653,-87.951911,8933 S 27th St,Franklin,"WI, 53132",desserts,new york style cheesecake,topped with raspberry sauce.,7.95
8633,2337,11,Zaza Steak and Lemonade,4.4,200.0,"burgers, american, sandwiches, chicken, kids f...",Económico,"9618 W .Fon Du Lac Ave, Milwaukee, WI, 53225",43.133201,-88.031668,9618 W .Fon Du Lac Ave,Milwaukee,"WI, 53225",sandwiches,southern bbq steak combo,"green peppers, onions, mushrooms, provolone ch...",11.99
5089,2219,27,Fryerz (W Fond du Lac Ave),3.8,171.0,"breakfast and brunch, american, sandwiches",No Registra,"2651 W Fond Du Lac Ave, Milwaukee, WI, 53206",43.067030,-87.947180,2651 W Fond Du Lac Ave,Milwaukee,"WI, 53206",family combo dinners,wings (21 pcs) &amp; fish (21 pcs),sin descripcion,72.99
47799,8991,224,Subway (Roosevelt),4.4,148.0,"american, sandwich, salads, vegetarian friendly",Moderadamente caro,"4336 Roosevelt Way NE, Seattle, WA, 98105",47.661008,-122.317350,4336 Roosevelt Way NE,Seattle,"WA, 98105",no bready bowls™,sweet onion chicken teriyaki,the amazing sweet onion chicken teriyaki prote...,11.75
39569,8439,24,Taqueria Tequila Authentic Mexican Food (Phinn...,4.7,200.0,"mexican, seafood, salads, healthy, family frie...",Económico,"301 NW 85th St., Seattle, WA, 98117",47.690398,-122.360973,301 NW 85th St.,Seattle,"WA, 98117",sides,sour cream,sin descripcion,2.99




FILAS      : 63,021
COLUMNAS   : 17
DUPLICADOS : 38


TIPOS DE DATOS


,restaurant_id,position,name_restaurant,score,ratings,category_restaurant,price_range,full_address,lat,lng,calle,ciudad,estado_zip,category_menu,name_menu,description,price
0,int64,int64,object,float64,float64,object,object,object,float64,float64,object,object,object,object,object,object,float64





CANTIDAD DE VALORES NULOS POR COLUMNA


,restaurant_id,position,name_restaurant,score,ratings,category_restaurant,price_range,full_address,lat,lng,calle,ciudad,estado_zip,category_menu,name_menu,description,price
Valores nulos,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,15,0





PORCENTAJE DE VALORES NULOS POR COLUMNA


,restaurant_id,position,name_restaurant,score,ratings,category_restaurant,price_range,full_address,lat,lng,calle,ciudad,estado_zip,category_menu,name_menu,description,price
% Valores nulos,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.02,0.0





CANTIDAD DE VALORES ÚNICOS POR COLUMNA


,restaurant_id,position,name_restaurant,score,ratings,category_restaurant,price_range,full_address,lat,lng,calle,ciudad,estado_zip,category_menu,name_menu,description,price
Valores únicos,669,212,662,14,98,433,4,654,667,667,653,64,111,2374,27759,22140,2023


## LIMPEIZA EN DATA FRAME MAESTRO

In [12]:
## identificar registros duplicados
df_master.duplicated().sum()

np.int64(38)

In [13]:
duplicados = df_master[df_master.duplicated(keep=False)]
duplicados.sort_values(by="restaurant_id")

,restaurant_id,position,name_restaurant,score,ratings,category_restaurant,price_range,full_address,lat,lng,calle,ciudad,estado_zip,category_menu,name_menu,description,price
4195,2200,51,Champion Chicken,4.2,103.0,"burgers, american, sandwiches, pizza",Económico,"8718 W Lisbon Ave, Milwaukee, WI, 53222",43.083239,-88.022317,8718 W Lisbon Ave,Milwaukee,"WI, 53222",side orders,pint of chicken flavored rice,sin descripcion,5.95
4196,2200,51,Champion Chicken,4.2,103.0,"burgers, american, sandwiches, pizza",Económico,"8718 W Lisbon Ave, Milwaukee, WI, 53222",43.083239,-88.022317,8718 W Lisbon Ave,Milwaukee,"WI, 53222",side orders,pint of chicken flavored rice,sin descripcion,5.95
8691,2337,11,Zaza Steak and Lemonade,4.4,200.0,"burgers, american, sandwiches, chicken, kids f...",Económico,"9618 W .Fon Du Lac Ave, Milwaukee, WI, 53225",43.133201,-88.031668,9618 W .Fon Du Lac Ave,Milwaukee,"WI, 53225",nachos and salads,chicken philly salad,chicken philly salad includes the following ch...,10.99
8695,2337,11,Zaza Steak and Lemonade,4.4,200.0,"burgers, american, sandwiches, chicken, kids f...",Económico,"9618 W .Fon Du Lac Ave, Milwaukee, WI, 53225",43.133201,-88.031668,9618 W .Fon Du Lac Ave,Milwaukee,"WI, 53225",nachos and salads,chicken philly salad,chicken philly salad includes the following ch...,10.99
10303,2404,9,7-Eleven (1624 W Wells St),4.5,200.0,"convenience, everyday essentials, grocery, sna...",Económico,"1624 W Wells St, Milwaukee, WI, 53233",43.040628,-87.933943,1624 W Wells St,Milwaukee,"WI, 53233",household,24/7 life premium bath tissue 4 count,our super soft premium bath tissue is great to...,3.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55308,9462,1,7-Eleven (14501 Juanita-Woodinville),4.9,136.0,"convenience, everyday essentials, snacks, home...",Económico,"14501 Juanita-Woodinville, Bothell, WA, 98011",47.733434,-122.195582,14501 Juanita-Woodinville,Bothell,"WA, 98011",fresh food,large pizza - cheese,100%25 real® mozzarella made from whole milk p...,7.99
55312,9462,1,7-Eleven (14501 Juanita-Woodinville),4.9,136.0,"convenience, everyday essentials, snacks, home...",Económico,"14501 Juanita-Woodinville, Bothell, WA, 98011",47.733434,-122.195582,14501 Juanita-Woodinville,Bothell,"WA, 98011",ice cream,edy's dibs crunch 4oz,"bites of crispy, chocolatey coating filled wit...",2.50
55314,9462,1,7-Eleven (14501 Juanita-Woodinville),4.9,136.0,"convenience, everyday essentials, snacks, home...",Económico,"14501 Juanita-Woodinville, Bothell, WA, 98011",47.733434,-122.195582,14501 Juanita-Woodinville,Bothell,"WA, 98011",ice cream,ben & jerry's half baked pint,vanilla ice cream with gobs of chocolate chip ...,7.49
55326,9462,1,7-Eleven (14501 Juanita-Woodinville),4.9,136.0,"convenience, everyday essentials, snacks, home...",Económico,"14501 Juanita-Woodinville, Bothell, WA, 98011",47.733434,-122.195582,14501 Juanita-Woodinville,Bothell,"WA, 98011",ice cream,edy's dibs crunch 4oz,"bites of crispy, chocolatey coating filled wit...",2.50


In [14]:
## generando un backup de la data maestra
df_master_bkp = df_master.copy()
## elimnando los registros duplicados
df_master_bkp = df_master_bkp.drop_duplicates()

In [15]:
## reemplazar valores nulos
df_master_bkp["description"] = (
    df_master_bkp["description"]
    .fillna("sin descripcion")
)

## CREAR MOTOR SQLite

SQLAlchemy crea un motor de conexión (database engine).

Ese objeto contiene toda la información necesaria para conectarse a la base de datos:

Tipo de base de datos (SQLite).
Nombre o ubicación de la base (delivery_insights.db).
La forma de abrir y cerrar conexiones.


In [29]:
## creando la base de datos
engine = create_engine(
    "sqlite:///delivery_insights.db"
)

## CARGA DE LA TABLA

In [30]:
df_master_bkp.to_sql(
    # nombre de la tabla
    name="master_food_data",
    # conexion a SQLite
    con=engine,
    # si la tabla existe, la elimina, vuelve a crear e inserta la data
    if_exists="replace",
    # no guarda el indice de pd como una columna
    index=False
)

62983

## VALIDANDO CARGA CORRECTA

In [31]:
## consultando tabla recien creada
consulta = pd.read_sql(
    "SELECT * FROM master_food_data LIMIT 5",
    con=engine # la bd se conecta a pd para realizar la operacion.
)

consulta

,restaurant_id,position,name_restaurant,score,ratings,category_restaurant,price_range,full_address,lat,lng,calle,ciudad,estado_zip,category_menu,name_menu,description,price
0,1739,21,Pho Cali Noodle House,4.4,200.0,"vietnamese, noodles, sandwich, asian",Económico,"4756 S 27th St, Milwaukee, WI, 53221",42.958143,-87.94815,4756 S 27th St,Milwaukee,"WI, 53221",picked for you,spring roll,"three rolls. shrimps, sliced pork, vermicelli ...",6.99
1,1739,21,Pho Cali Noodle House,4.4,200.0,"vietnamese, noodles, sandwich, asian",Económico,"4756 S 27th St, Milwaukee, WI, 53221",42.958143,-87.94815,4756 S 27th St,Milwaukee,"WI, 53221",picked for you,egg roll,"pork, carrot, bean thread noodles, and are wra...",6.50
2,1739,21,Pho Cali Noodle House,4.4,200.0,"vietnamese, noodles, sandwich, asian",Económico,"4756 S 27th St, Milwaukee, WI, 53221",42.958143,-87.94815,4756 S 27th St,Milwaukee,"WI, 53221",picked for you,grilled pork,foot long.,9.50
3,1739,21,Pho Cali Noodle House,4.4,200.0,"vietnamese, noodles, sandwich, asian",Económico,"4756 S 27th St, Milwaukee, WI, 53221",42.958143,-87.94815,4756 S 27th St,Milwaukee,"WI, 53221",picked for you,house special,"rice noodle with steak, rare, flank, brisket, ...",10.95
4,1739,21,Pho Cali Noodle House,4.4,200.0,"vietnamese, noodles, sandwich, asian",Económico,"4756 S 27th St, Milwaukee, WI, 53221",42.958143,-87.94815,4756 S 27th St,Milwaukee,"WI, 53221",picked for you,rice noodle with rare steak,sin descripcion,10.95


In [32]:
## cantidad de registros de la tabla
pd.read_sql(
    """
    SELECT COUNT(*) AS total_registros
    FROM master_food_data
    """,
    con=engine
)

,total_registros
0,62983


In [33]:
## con python
print(len(df_master_bkp))

62983


## DESCARGA DE LA BD

In [35]:
files.download("/content/delivery_insights.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>